# Visualizing enVision data

In [1]:
import polars as pl
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
import string
from tqdm import tqdm
import numpy as np
from scipy.signal import savgol_filter

rows = list(string.ascii_uppercase[:16])  # A-P
cols = list(range(1, 25))                 # 1-24

wells = [f"{r}{c:02d}" for r in rows for c in cols]


dataset_path = Path(
    "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
    "Study/enVision/enVision_preprocessed/2026-04-01._."
    "PREPROCESSED.csv"
    )

raw_data_path = Path(
    "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live"
    " Study/enVision/enVision_formatted/2026-04-01._.FORMATTED.tsv"
    )

platemap = (
    pl.read_excel(
    "/home/derfelt/PartchLabFiles/Circadian"
    " Rhythms/Live Study/2026-04-01Echo_transfer.xlsx", engine="openpyxl")
    .select(["384_Well", 
             "MoleculeID", 
             "strain",
             "dex"])
    .unique(subset=["384_Well"])
    .with_columns(
        pl.col("MoleculeID").str.strip_chars()
    )
    )

well_excluded = set(wells) - set(platemap["384_Well"].to_list())

print(f"Number of wells not in platemap: {len(well_excluded)}")
# print(*well_excluded, sep="\n")

display(platemap)
# for name, df in platemap.group_by(["MoleculeID", "strain", "dex"]):
#     print(name)
#     display(df)
    # if name[0] == "LY 5uM":
    #     print(name)
    #     display(df)

Number of wells not in platemap: 146


384_Well,MoleculeID,strain,dex
str,str,str,str
"""D03""","""MK-1804 50uM""","""Bmal1""","""dex"""
"""I22""","""MK-8692 50uM""","""Per2""","""dex"""
"""K14""","""MK-1804 25uM""","""Per2""","""dex"""
"""K21""","""MK-1496 50uM""","""Per2""","""dex"""
"""E21""","""LY 10uM""","""Bmal1""","""dex"""
…,…,…,…
"""J14""","""0.5% DMSO""","""Per2""","""NoSynhronisation"""
"""E17""","""S.pneumo#6""","""Bmal1""","""dex"""
"""L21""","""MK-2267 100uM""","""Per2""","""dex"""


In [2]:
def make_title(row: dict, well_col: str, meta_cols: list[str]) -> str:
    parts = [row[well_col]]
    for col in meta_cols:
        val = row.get(col)
        if val is not None and str(val).strip() != '' and str(val).lower() != 'nan':
            parts.append(str(val))
    return ' | '.join(parts)

In [3]:
raw_timeseries = pl.read_csv(raw_data_path, separator="\t")
raw_timeseries = raw_timeseries.join(platemap, left_on="Well", right_on="384_Well", how="left")

wells = raw_timeseries["Well"].unique().sort().to_list()
out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/enVision_visual_results/raw_timeseries.pdf")

timepoint_col = "Timepoint"
value = "CalcResultI"
meta_cols = ['MoleculeID', 'strain', 'dex']

with PdfPages(out_path) as pdf:
    g = sns.FacetGrid(
        data=raw_timeseries,
        col="Well",
        col_wrap=24,
        height=2.5,
        aspect=1.8,
        sharey=False,
        sharex=True,
    )
    g.map_dataframe(sns.lineplot, x=timepoint_col, y=value, color="red", linewidth=1.5)
    
    # add marks and set ticks after mapping
    x_min = raw_timeseries['Timepoint'].min()
    x_max = raw_timeseries['Timepoint'].max()
    tick_marks = np.arange(np.ceil(x_min / 12) * 12, x_max + 12, 12)

    for ax in g.axes.flat:
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_xticks(tick_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in tick_marks], fontsize=6)
        for mark in tick_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        
        
    g.set_titles(col_template='Well {col_name}')
    g.set_axis_labels('Time (hours)', 'Detrended signal')
    g.tight_layout()
    pdf.savefig(g.figure)
    plt.close()
    
    
    for well in tqdm(wells, total=len(wells), desc="Well progress"):
        well_df = (raw_timeseries
              .filter((pl.col("Well") == well))
              .sort("Timepoint", descending=False)
              .to_pandas()
              )
        
        fig, ax = plt.subplots(figsize=(12, 4))
        
        # get first row for metadata
        row = well_df.iloc[0].to_dict()
        title = make_title(row, well_col='Well', meta_cols=meta_cols)
        
        sns.lineplot(data=well_df, x=timepoint_col, y=value, ax=ax, color="red", linewidth=2, markers=False, marker="o", markersize=2)
        
        ax.set_title(title)
        ax.set_xlabel("Timepoint")
        ax.set_ylabel("Expression")
        
        x_min = well_df[timepoint_col].min()
        x_max = well_df[timepoint_col].max()
        period_marks = np.arange(np.ceil(x_min/12) * 12, x_max + 12, 12)
        
        for mark in period_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        ax.set_xticks(period_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in period_marks])
        
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        

Well progress: 100%|██████████| 384/384 [00:33<00:00, 11.60it/s]


## We now need to do the same exact thing but for detrended data

In [4]:
dataset = (
    pl.read_csv(dataset_path)
    .filter(pl.col("feature") == "CalcResultI")
    .select(["Well", "Timepoint", "mean"])
    .join(platemap, left_on="Well", right_on="384_Well", how="left")
    .sort(['Well', 'Timepoint'])
    )

# smoothing option
dataset = (dataset
           .group_by("Well")
           .map_groups(
               lambda g: 
                   g.with_columns(
                       pl.Series("mean", savgol_filter(
                           g["mean"].to_numpy(),
                           window_length=13,
                           polyorder=3
                       ))
                       )
                   )
           .sort(["Well", "Timepoint"])
)

display(dataset)

wells = dataset["Well"].unique().sort().to_list()
out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/enVision_visual_results/detrended_savgol_smoothed_timeseries.pdf")

timepoint_col = "Timepoint"
value = "mean"
meta_cols = ['MoleculeID', 'strain', 'dex']

with PdfPages(out_path) as pdf:
    g = sns.FacetGrid(
        data=dataset,
        col="Well",
        col_wrap=24,
        height=2.5,
        aspect=1.8,
        sharey=False,
        sharex=True,
    )
    g.map_dataframe(sns.lineplot, x=timepoint_col, y=value, color="red", linewidth=1.5)
    
    # add marks and set ticks after mapping
    x_min = dataset['Timepoint'].min()
    x_max = dataset['Timepoint'].max()
    tick_marks = np.arange(np.ceil(x_min / 12) * 12, x_max + 12, 12)

    for ax in g.axes.flat:
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_xticks(tick_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in tick_marks], fontsize=6)
        for mark in tick_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        
        
    g.set_titles(col_template='Well {col_name}')
    g.set_axis_labels('Time (hours)', 'Detrended signal')
    g.tight_layout()
    pdf.savefig(g.figure)
    plt.close()
    
    
    for well in tqdm(wells, total=len(wells), desc="Well progress"):
        well_df = (dataset
              .filter((pl.col("Well") == well))
              .sort("Timepoint", descending=False)
              .to_pandas()
              )
        
        fig, ax = plt.subplots(figsize=(12, 4))
        
        # get first row for metadata
        row = well_df.iloc[0].to_dict()
        title = make_title(row, well_col='Well', meta_cols=meta_cols)
        
        # sns.lineplot(data=well_df, x=timepoint_col, y=value, ax=ax, color="red", linewidth=2 marker="o", markersize=2)
        sns.lineplot(data=well_df, x=timepoint_col, y=value, ax=ax, color="red", linewidth=2)
        
        ax.set_title(title)
        ax.set_xlabel("Timepoint")
        ax.set_ylabel("Expression")
        
        x_min = well_df[timepoint_col].min()
        x_max = well_df[timepoint_col].max()
        period_marks = np.arange(np.ceil(x_min/12) * 12, x_max + 12, 12)
        
        for mark in period_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        ax.set_xticks(period_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in period_marks])
        
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        

Well,Timepoint,mean,MoleculeID,strain,dex
str,f64,f64,str,str,str
"""A01""",24.0,537.077014,null,null,null
"""A01""",24.5,544.336271,null,null,null
"""A01""",25.0,532.842249,null,null,null
"""A01""",25.5,504.881559,null,null,null
"""A01""",26.0,462.740816,null,null,null
…,…,…,…,…,…
"""P24""",92.5,389.77274,null,null,null
"""P24""",93.0,373.949479,null,null,null
"""P24""",93.5,325.039232,null,null,null


Well progress: 100%|██████████| 384/384 [00:28<00:00, 13.52it/s]


## Now we do this via grouping the meta data and reporting a mean and sd

In [8]:
dataset = (
    pl.read_csv(dataset_path)
    .filter(pl.col("feature") == "CalcResultI")
    .select(["Well", "Timepoint", "mean"])
    .join(platemap, left_on="Well", right_on="384_Well", how="left")
    .sort(['Well', 'Timepoint'])
    )

# smoothing option
dataset = (dataset
           .group_by("Well")
           .map_groups(
               lambda g: 
                   g.with_columns(
                       pl.Series("mean", savgol_filter(
                           g["mean"].to_numpy(),
                           window_length=13,
                           polyorder=3
                       ))
                       )
                   )
           .sort(["dex","strain","MoleculeID", "Well", "Timepoint"], descending=[True, True, True, False, False])
)

wells = dataset["Well"].unique().sort().to_list()
out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/enVision_visual_results/grouped_results.pdf")

timepoint_col = "Timepoint"
value = "mean"
meta_cols = ['MoleculeID', 'strain', 'dex']

dataset = dataset.to_pandas()
dataset['group_label'] = dataset[meta_cols].apply(
    lambda row: ' | '.join(str(v) for v in row), axis=1
)

groups = dataset['group_label'].unique()
print(*groups, sep="\n")

x_min = dataset[timepoint_col].min()
x_max = dataset[timepoint_col].max()
tick_marks = np.arange(np.ceil(x_min/12) * 12, x_max + 12, 12)

with PdfPages(out_path) as pdf:
    
    # facet
    
    g = sns.FacetGrid(dataset, col="group_label", col_wrap=10, height=3, aspect=2, sharey=False, sharex=True)
    g.map_dataframe(sns.lineplot, x=timepoint_col, y=value, errorbar='sd', color='red', linewidth=1.5, 
                    err_kws={'alpha': 0.5, 'facecolor': 'red'})

    for ax in g.axes.flat:
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_xticks(tick_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in tick_marks], fontsize=6)
        for mark in tick_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    g.set_titles(col_template='{col_name}')
    g.set_axis_labels('Time (hours)', 'Deterended and Smoothed Expression')
    g.tight_layout()
    pdf.savefig(g.figure)
    plt.close()
    
    for group in tqdm(groups, desc='plotting groups'):
        gdf = dataset[dataset['group_label'] == group]
        
        fig, ax = plt.subplots(figsize=(12,4))
        sns.lineplot(
            data=gdf,
            x=timepoint_col,
            y=value,
            errorbar='sd',
            color='red',
            linewidth=1.5,
            ax=ax,
            err_kws={'alpha': 0.2, 'facecolor': 'red'}
        )
        
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_title(group)
        ax.set_xlabel('Time (hours)')
        ax.set_ylabel('Deterended and Smoothed Expression')
        
        x_min_g = gdf['Timepoint'].min()
        x_max_g = gdf['Timepoint'].max()
        period_marks = np.arange(np.ceil(x_min_g / 12) * 12, x_max_g + 12, 12)
        
        for mark in period_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        ax.set_xticks(period_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in period_marks])

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        
# display(dataset)

nan | nan | nan
food_coloring | nan | dex
T0901317 5uM | Per2 | dex
T0901317 10uM | Per2 | dex
SR 5uM | Per2 | dex
SR 10uM | Per2 | dex
PF 1uM | Per2 | dex
Nob 50uM | Per2 | dex
Nar 50uM | Per2 | dex
MK-8800 50uM | Per2 | dex
MK-8800 25uM | Per2 | dex
MK-8800 100uM | Per2 | dex
MK-8692 50uM | Per2 | dex
MK-8692 25uM | Per2 | dex
MK-8692 100uM | Per2 | dex
MK-2267 25uM | Per2 | dex
MK-2267 100uM | Per2 | dex
MK-1804 50uM | Per2 | dex
MK-1804 25uM | Per2 | dex
MK-1804 100uM | Per2 | dex
MK-1496 50uM | Per2 | dex
MK-1496 25uM | Per2 | dex
MK-1496 100uM | Per2 | dex
MK-1282 50uM | Per2 | dex
MK-1282 25uM | Per2 | dex
MK-1282 100uM | Per2 | dex
MK-0769 50uM | Per2 | dex
MK-0769 25uM | Per2 | dex
MK-0769 100uM | Per2 | dex
MK-0761 50uM | Per2 | dex
MK-0761 25uM | Per2 | dex
MK-0761 100uM | Per2 | dex
MH-23 50uM | Per2 | dex
MH-23 25uM | Per2 | dex
MH-23 100uM | Per2 | dex
MH-20 50uM | Per2 | dex
MH-20 25uM | Per2 | dex
MH-20 100uM | Per2 | dex
MH-13 25uM | Per2 | dex
MH-13 100uM | Per2 | dex

plotting groups: 100%|██████████| 129/129 [00:12<00:00, 10.65it/s]
